In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from csv import DictReader
from os import makedirs, path, listdir
from markitdown import MarkItDown
from requests import get
from gitsource import chunk_documents
from collections import Counter
from minsearch import Index


# Create a directory if it does not already exist
def create_directory(directory_name):
    if not path.exists(directory_name):
        makedirs(directory_name)

# Extract the file name from the PDF URL
# Example: https://example.com/book.pdf -> book.pdf
def get_file_name(pdf_url):
    return pdf_url.split("/")[-1]


def download_books(csv_response, books_dir):
    """
    Download PDF books listed in a CSV response into the specified directory.

    The CSV is expected to contain 'title' and 'pdf_url' columns.
    Existing PDF files are skipped to avoid downloading them again.
    """
    # Convert the CSV text into rows that can be read by column name
    reader = DictReader(csv_response.text.splitlines())

    # Go through every book in the CSV file
    for row in reader:
        title = row["title"]
        pdf_url = row["pdf_url"]

        # Get the PDF file name from the URL
        file_name = get_file_name(pdf_url)

        # Build the local path where the PDF will be saved
        local_path = f"{books_dir}/{file_name}"

        # Skip the download if this book already exists
        if path.exists(local_path):
            print(f"Already exists: {title}")
            continue

        print(f"Downloading: {title}")

        # Download the PDF file
        pdf_response = get(pdf_url)
        pdf_response.raise_for_status()

        # Save the PDF in binary mode
        with open(local_path, "wb") as file:
            file.write(pdf_response.content)

        print(f"Downloaded: {title}")


def convert_books_to_markdown(books_dir, books_text_dir):
    """
    Convert all PDF books in the source directory into Markdown files.

    Only PDF files are processed. Existing Markdown files are skipped
    to avoid converting the same book more than once.
    """
    markitdown = MarkItDown()

    # Go through all files in the books directory
    for file_name in listdir(books_dir):

        # Process PDF files only
        if not file_name.endswith(".pdf"):
            continue

        pdf_path = f"{books_dir}/{file_name}"

        # Change .pdf extension to .md
        markdown_file_name = file_name.replace(".pdf", ".md")
        markdown_path = f"{books_text_dir}/{markdown_file_name}"

        # Skip conversion if the Markdown file already exists
        if path.exists(markdown_path):
            print(f"Already exists: {markdown_file_name}")
            continue

        print(f"Converting: {file_name}")

        # Convert PDF to Markdown
        result = markitdown.convert(pdf_path)

        # Save the extracted Markdown text
        with open(markdown_path, "w", encoding="utf-8") as file:
            file.write(result.markdown)

        print(f"Saved: {markdown_path}")
    
def prepare_documents(books_text_dir):
    """
    Read Markdown book files and prepare them as structured documents.

    Each Markdown file is converted into a dictionary containing the
    source file name and a list of non-empty text lines.
    """
    documents = []

    for file_name in listdir(books_text_dir):

        if not file_name.endswith(".md"):
            continue

        file_path = f"{books_text_dir}/{file_name}"

        with open(file_path, "r", encoding="utf-8") as file:
            lines = file.readlines()

        # Remove empty lines and lines with whitespace only
        content = [line.strip() for line in lines if line.strip()]

        # Create one dictionary for each book
        document = {
            "source": file_name,
            "content": content
        }

        documents.append(document)

    return documents


def chunk_books(documents, size=100, step=50):
    """
    Split book documents into smaller overlapping text chunks.

    The chunk size controls how much text is included in each chunk,
    while the step controls how far the window moves between chunks.
    """
    # Chunk all books using the gitsource sliding window
    chunks = chunk_documents(
        documents,
        size=size,
        step=step
    )

    return chunks


def print_chunk_count(chunks):
    # Count chunks for each source file
    chunk_count = Counter(chunk["source"] for chunk in chunks)

    for source, count in chunk_count.items():
        print(f"{source}: {count} chunks")


def prepare_documents_for_index(chunks):
    """
    Prepare text chunks for indexing.

    Each chunk is converted into a dictionary containing the source
    file name and its content joined into a single text string.
    """
    documents = []

    for chunk in chunks:
        document = {
            "source": chunk["source"],
            "content": "\n".join(chunk["content"])
        }

        documents.append(document)

    return documents

load_dotenv(override=True)

openai_client = OpenAI()

In [2]:
# GitHub repository information
repo_site = "https://raw.githubusercontent.com"
repo_owner = "alexeygrigorev"
repo_name = "ai-engineering-buildcamp-code"
branch = "main"
file_path = "01-foundation/homework/books.csv"

# Build the full URL of the CSV file
file_url = f'{repo_site}/{repo_owner}/{repo_name}/{branch}/{file_path}'

In [3]:
books_dir = "books"
books_text_dir = "books_text"

In [4]:
create_directory(books_dir)
create_directory(books_text_dir)

In [5]:
csv_response = get(file_url)
csv_response.raise_for_status()

In [ ]:
# task 1: Download books from the CSV file and save them in the books directory

download_books(csv_response, books_dir)

convert_books_to_markdown(books_dir, books_text_dir)

Downloading: Think Python 2e
Downloaded: Think Python 2e
Downloading: Think DSP
Downloaded: Think DSP
Downloading: Think Complexity 2e
Downloaded: Think Complexity 2e
Downloading: Think Java 2e
Downloaded: Think Java 2e
Downloading: Physical Modeling in MATLAB
Downloaded: Physical Modeling in MATLAB
Downloading: Think OS
Downloaded: Think OS
Downloading: Think C++
Downloaded: Think C++
Converting: thinkcomplexity2.pdf
Saved: books_text/thinkcomplexity2.md
Converting: thinkpython2.pdf
Saved: books_text/thinkpython2.md
Converting: PhysicalModelingInMatlab4.pdf
Saved: books_text/PhysicalModelingInMatlab4.md
Converting: thinkjava2.pdf
Saved: books_text/thinkjava2.md
Converting: thinkdsp.pdf
Saved: books_text/thinkdsp.md
Converting: thinkos.pdf
Saved: books_text/thinkos.md
Converting: Think-C.pdf
Saved: books_text/Think-C.md


In [ ]:
 #task 2: Prepare documents and chunk them for indexing
books_text_dir = "books_text"

# Prepare all Markdown books
documents = prepare_documents(books_text_dir)

# Chunk all books
chunks = chunk_books(
    documents,
    size=100,
    step=50
)

# Print the number of chunks for each book
print_chunk_count(chunks)

Think-C.md: 107 chunks
PhysicalModelingInMatlab4.md: 119 chunks
thinkos.md: 68 chunks
thinkcomplexity2.md: 144 chunks
thinkdsp.md: 111 chunks
thinkjava2.md: 250 chunks
thinkpython2.md: 189 chunks


In [ ]:
 # task 3: Prepare documents for indexing and create an index

documents = prepare_documents_for_index(chunks)

index = Index(text_fields=["content"])
index.fit(documents)

print(f"Indexed documents: {len(documents)}")

Indexed documents: 988


In [ ]:
# task 4: Search the index for relevant content
results = index.search(
    "python function definition",
    num_results=5
)

for result in results:
    print(result["source"])
    print(result["content"])
    print("-" * 80)

PhysicalModelingInMatlab4.md
| ------ | ------------ | --- | --- |
| total  | = total +    | a;  |     |
end
ans = total
Ifyouwereusinganyofthosevariablenamesbeforecallingthisscript, youmightbesurprised
to find, after running the script, that their values had changed. If you have two scripts that
48 Functions
use the same variable names, you might find that they work separately and then break when
you try to combine them. This kind of interaction is called a collision.
name
As the number of scripts you write increases, and they get longer and more complex, name
collisions become more of a problem. Avoiding this problem is one of several motivations for
functions.
| 5.2 | Defining | Functions |     |     |
| --- | -------- | --------- | --- | --- |
A function is like a script, except that each function has its own workspace, so any variables
defined inside a function only exist while the function is running and don’t interfere with
variables in other workspaces, even if they have the sa

In [ ]:
# task 5: Full Generative Retrieval-Augmented Generation (RAG) pipeline
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return {
        "answer": response.output_text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens
    }

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    result = llm(prompt, instructions)

    return result



answer = rag("python function definition")

print(answer)

{'answer': 'A Python function is defined using the `def` keyword, followed by the function name and parentheses containing any parameters. Here’s a general structure:\n\n```python\ndef function_name(parameters):\n    # function body\n    return result\n```\n\n### Example:\nHere’s an example of a simple Python function that calculates the sum of two numbers:\n\n```python\ndef add_numbers(a, b):\n    result = a + b\n    return result\n```\n\n### Calling the Function:\nYou can call this function and print the result like this:\n\n```python\nsum_result = add_numbers(5, 3)\nprint(sum_result)  # Output: 8\n```\n\n### Key Points:\n- Functions can take multiple input parameters.\n- The `return` statement is used to output a value from the function.\n- The function has its own scope, meaning local variables defined inside the function are not accessible outside it.\n\nThis structure helps prevent name collisions and keeps your workspace organized, similar to what is described in the provided co

In [ ]:
# task 6: Structured vs Unstructured Output
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [15]:
def llm2(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=RAGResponse
    )

    return {
        "answer": response.output_parsed,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens
    }

In [16]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    result = llm2(prompt, instructions)

    return result

structured_result = rag("python function definition")

print(structured_result["answer"])
print("Input tokens:", structured_result["input_tokens"])
print("Output tokens:", structured_result["output_tokens"])

unstructured_result = answer

more_tokens = (
    structured_result["input_tokens"]
    - unstructured_result["input_tokens"]
)

print("Q5 input tokens:", unstructured_result["input_tokens"])
print("Q6 input tokens:", structured_result["input_tokens"])
print("More input tokens:", more_tokens)

answer='In Python, a function is defined using the `def` keyword followed by the function name and parentheses containing any parameters. Here’s a basic structure of how it looks:\n\n```python\ndef function_name(parameters):\n    # Function body\n    # Code to execute\n    return output_variable\n```\n\n### Example:\nHere is an example of a function that calculates the square of a number:\n\n```python\ndef square(x):\n    return x ** 2\n```\n\n### Explanation:\n- **def**: This keyword signifies the beginning of the function definition.\n- **function_name**: Replace with the name you want to give your function.\n- **parameters**: Input values to the function (optional, can be none).\n- **function body**: Indented code block where you implement the logic of the function.\n- **return statement**: Ends the function and specifies the value to return.\n\n### Calling a Function:\nTo use or "call" the function, simply write its name followed by arguments in parentheses, like this:\n\n```python